In [1]:
import requests
import pandas as pd
import time
import ast

##### Fetch all documents for the specified session

In [59]:
def get_documents(session_number: int):
    """
    Obtain all documents for the specified session of the General Court
    """
    # Returns all of documents publicly available by the General Court
    url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Documents/"
    print(f"\n Fetching documents from General Court {session_number} ...")

    response = requests.get(url)
    response.raise_for_status()
    docs = response.json()
    print(f"Successfully retrieved {len(docs)} documents from General Court {session_number}.")
    
    return docs


In [60]:
# docs_193 = get_documents(193)

# for i, doc in enumerate(docs_193[:3]):
#     print(f"\n--- Document {i+1} preview ---")
#     print(doc)


##### Extract information from documents

In [61]:
import pandas as pd

def extract_bill_info(docs_json, session_number):
    """
    Extract key information from the documents JSON of the Legislature API 
    and convert it into DataFrame format.
    
    Return:
        pd.DataFrame: 
        containing billnumbers, docketnumbers, title, branch, and IsDocketBookOnly
    """
    bill_list = []

    for doc in docs_json:
        bill_no = doc.get("BillNumber")
        docket_no = doc.get("DocketNumber")
        title = doc.get("Title")
        is_book_only = doc.get("IsDocketBookOnly")

        bill_list.append({
            "billNumber": bill_no if bill_no else None,
            "docketNumber": docket_no if docket_no else None,
            "title": title if title else None,
            "branch": (
                "Senate" if bill_no and bill_no.startswith("S") else
                "House" if bill_no and bill_no.startswith("H") else None
            ),
            "isDocketBookOnly": is_book_only
        })

    df = pd.DataFrame(bill_list)
    print(f"Extracted {len(df)} entries from documents for General Court {session_number if session_number else ''}...")
    #print("Preview of first 5 rows:")
    #print(df.head())
    
    print("\nMissing value summary:")
    print(df.isna().sum())

    print("\n Count by branch:")
    print(df['branch'].value_counts(dropna=False))

    return df


# df_193 = extract_bill_info(docs_193, 193)


##### Filter bills containing rollcall records

In [62]:

def get_rollcall_status(df_bills, session_number=None):
    """
    Scan each bill to determine whether a roll call vote record exists.
    """

    # Filter bills with billnumbers：billnumber is not null + Senate bills（the API does not return House records)
    bill_numbers = df_bills[(df_bills['billNumber'].notna()) &
                            (df_bills['branch'] == 'Senate')]['billNumber'].tolist()
    total = len(bill_numbers)
    rollcall_records = []

    print(f"\nStarting roll call scan for General Court {session_number if session_number else ''}")
    print(f"Total bills to scan: {total}\n")

    for index, bill in enumerate(bill_numbers):
        url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Documents/{bill}/RollCalls"
        

        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            # Extract roll call number
            if isinstance(data, list) and len(data) > 0:
                roll_nums = [item.get("RollCallNumber") for item in data if "RollCallNumber" in item]
            else:
                roll_nums = []

            rollcall_records.append({
                "billNumber": bill,
                "rollCallNumbers": roll_nums,
                "hasRollCall": len(roll_nums) > 0
            })

        except Exception as e:
            rollcall_records.append({
                "billNumber": bill,
                "rollCallNumbers": None,
                "hasRollCall": None
            })
            #print(f"Error fetching {bill}: {e}")

        if (index + 1) % 500 == 0 or (index + 1) == total:
            print(f"📍 Progress: {index + 1}/{total} bills processed...")

        #time.sleep(delay)

    df_rollcall = pd.DataFrame(rollcall_records)

    print("\nRoll Call Scan Summary:")
    print(df_rollcall['hasRollCall'].value_counts(dropna=False))
    print("\nExample preview:")
    print(df_rollcall.head())
    print(df_rollcall.shape)

    return df_rollcall

# df_bills_193 = get_rollcall_status(df_193, session_number=193)


##### Amendment Roll Call Mapping

In [63]:

df_bills_194 = pd.read_csv("C:\\Users\\jhend\\OneDrive\\Documents\\GitHub\\ds-act-on-mass\\fa25-team-b\\data\\194_senate_bill_full_details.csv")

In [64]:
#This converts the AmendmentNumbers column from string to list if they are a string for some reason
def literal_return(val):
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError) as e:
        return val

In [65]:
#This currently does not return general court.
def AmendmentRollCallMapping(df_bills,session_number):
    try:
        #Convert amendmentnumbers to list if they are strings
        df_bills['AmendmentNumbers'] = df_bills['AmendmentNumbers'].apply(literal_return)
    except:
        ...
    rollCallMapping = pd.DataFrame(columns = ["billNumber", "AmendmentNumbers", "RollCallNumbers"])
    
    for i in df_bills["billNumber"]:
        a = df_bills.loc[df_bills["billNumber"] == i]["billNumber"].index
        
        print("Bill: " + str(i) + " at time: " + time.strftime("%H:%M:%S"))   
        try:
            #I will assume amendment numbers are supposed to go from 1 to len. so just doing a repeat by adding a +1 to the range as 0 doesn't exist.
            for j in range(len(df_bills["AmendmentNumbers"][a[0]])+1):
                bill = i
                #currently only trying senate data.
                
                url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Documents/{bill}/Branches/Senate/Amendments/{j}"
                #print(url)
                try:
                    resp = requests.get(url, timeout=10)
                except requests.RequestException as e:
                    #print(f"Error fetching {bill} Amendment {j}: {e}")
                    continue
                
                if resp.status_code == 200:
                    data = resp.json()
                else:
                    #print(f"Failed to retrieve {bill} Amendment {j}: {resp.status_code}")
                    continue
                try:
                    roll_call_number = data["RollCall"][0]["RollCallNumber"]
                    rollCallMapping = pd.concat([rollCallMapping, pd.DataFrame({"billNumber": [i], "AmendmentNumbers": [j], "RollCallNumbers": [roll_call_number]})], ignore_index=True)
                except:
                    continue
                    #This means there are no rollcalls for this amendment
                
        except:
            #print("No amendments found for bill: " + str(i))
            continue
            
    return rollCallMapping
#amendmentRollCallMapping = AmendmentRollCallMapping(df_bills = df_bills_194, session_number = 194)
#print(amendmentRollCallMapping)

In [66]:
#amendmentRollCallMapping.to_csv("194_senate_amendment_rollcall_mapping.csv", index=False)
# I will note, Some roll calls are on items within a house bill and not amendments so they will not be captured here nor votes on primary bill.

##### Fetch the details of each vote by every member

In [67]:
def fetch_bill_vote_details(df_roll_status, session_number=None, delay=0.2):
    """
    Retrieve all voting details for legislators who participated in roll call votes 
    from the existing bill roll call status table.
    """
    # Select only bills that have had roll call votes.
    df_with_rolls = df_roll_status[df_roll_status['hasRollCall'] == True].copy()
    total_bills = len(df_with_rolls)
    print(f"\nFetching vote details for {total_bills} bills with roll calls "
          f"in General Court {session_number if session_number else ''}...\n")

    all_vote_records = []

    # Iterate through each bill with a roll call
    for idx, row in df_with_rolls.iterrows():
        bill_no = row['billNumber']
        roll_calls = row['rollCallNumbers'] if isinstance(row['rollCallNumbers'], list) else []
        if not roll_calls:
            continue

        for roll_num in roll_calls:
            url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Branches/Senate/RollCalls/{roll_num}"
            try:
                resp = requests.get(url, timeout=8)
                resp.raise_for_status()
                data = resp.json()

                # Extract the list of votes
                yea_list = data.get("Yeas", [])
                nay_list = data.get("Nays", [])
                absent_list = data.get("Absent", [])

                for vote_list, label in [(yea_list, "Yea"), (nay_list, "Nay"), (absent_list, "Absent")]:
                    for member in vote_list:
                        all_vote_records.append({
                            "GeneralCourtNumber": data.get("GeneralCourtNumber"),
                            "Branch": data.get("Branch"),
                            "RollCallNumber": roll_num,
                            "QuestionMotion": data.get("QuestionMotion"),  
                            "BillNumber": bill_no,
                            "MemberCode": member.get("MemberCode"),
                            "Vote": label,
                            "MemberUrl": member.get("Details")
                        })

                #print(f"RollCall {roll_num} from {bill_no}: "
                      #f"{len(yea_list)}Y / {len(nay_list)}N / {len(absent_list)}A")

            except Exception as e:
                print(f"Error fetching {bill_no} - RollCall {roll_num}: {e}")

            time.sleep(delay)  

    df_votes = pd.DataFrame(all_vote_records)
    print(f"\n Completed. Total vote records fetched: {len(df_votes)}")
    return df_votes


#df_vote_details_193 = fetch_bill_vote_details(df_bills_193, session_number=193, delay=0.1)

In [68]:
#df_vote_details_193.head()

In [69]:
#df_with_rolls = df_bills_193[df_bills_193['hasRollCall'] == True].copy()
#df_with_rolls

##### Fetch legislators info for the specified session

In [70]:
def fetch_legislators_from_votes(df_vote_details, session_number, delay=0.2):
    """
    Extract all MemberCodes from the vote details table,
    and call the API to retrieve detailed information on all legislators for that session.
    """

    # Extract all unique MemberCodes
    unique_codes = df_vote_details["MemberCode"].dropna().unique().tolist()
    total = len(unique_codes)
    print(f"\nFound {total} unique legislators in General Court {session_number}.\n")

    records = []

    # Iterate through each legislator and request data
    for idx, code in enumerate(unique_codes, start=1):
        url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/LegislativeMembers/{code}"
        try:
            resp = requests.get(url, timeout=12)
            resp.raise_for_status()
            data = resp.json()

            record = {
                "MemberCode": data.get("MemberCode"),
                "Name": data.get("Name"),
                "Branch": data.get("Branch"),
                "Party": data.get("Party"),
                "District": data.get("District"),
                "LeadershipPosition": data.get("LeadershipPosition"),
                "EmailAddress": data.get("EmailAddress"),
                "PhoneNumber": data.get("PhoneNumber"),
                "RoomNumber": data.get("RoomNumber"),
            }
            records.append(record)
            #print(f"{idx}/{total} {code} — {record['Name']}")

        except Exception as e:
            print(f"Error fetching {code}: {e}")
            continue

        time.sleep(delay)

    df_legislators = pd.DataFrame(records)
    print(f"\nRetrieved {len(df_legislators)} legislators for General Court {session_number}.")
    
    return df_legislators


In [71]:
#df_legislators_193 = fetch_legislators_from_votes(df_vote_details_193, session_number=193, delay=1)

#df_legislators_193.head()


In [72]:
def build_session_data(session_number, delay_rollcall=0.1, delay_vote=0.1, delay_legislator=0.2):
    """
    Enter the General Court session number and execute the complete data extraction process step by step.
    Return the voting details table and member information table for that session.
    """

    print(f"\nBuilding data for General Court {session_number} ...")

    # Step 1: extract documents
    docs_json = get_documents(session_number)
    df_bills = extract_bill_info(docs_json, session_number)

    # Step 2: extract roll call status
    df_roll_status = get_rollcall_status(df_bills, session_number=session_number)

    # Step 3: extract roll call vote details
    df_vote_details = fetch_bill_vote_details(
        df_roll_status,
        session_number=session_number,
        delay=delay_vote
    )

    # Step 4: extract legislators info
    df_legislators = fetch_legislators_from_votes(
        df_vote_details,
        session_number=session_number,
        delay=delay_legislator
    )

    print(f"Completed session {session_number}.")
    print(f"   Votes: {len(df_vote_details)} records, Legislators: {len(df_legislators)} entries.\n")

    return df_vote_details, df_legislators


In [ ]:
df_vote_details_193, df_legislators_193 = build_session_data(session_number=193)



Building data for General Court 193 ...

 Fetching documents from General Court 193 ...


In [ ]:
df_vote_details_192, df_legislators_192 = build_session_data(session_number=192)


Building data for General Court 192 ...

 Fetching documents from General Court 192 ...
Successfully retrieved 10363 documents from General Court 192.
Extracted 10363 entries from documents for General Court 192...

Missing value summary:
billNumber          1718
docketNumber        1295
title                  0
branch              1720
isDocketBookOnly       0
dtype: int64

 Count by branch:
branch
House     5462
Senate    3181
None      1720
Name: count, dtype: int64

Starting roll call scan for General Court 192
Total bills to scan: 3181

📍 Progress: 500/3181 bills processed...
📍 Progress: 1000/3181 bills processed...
📍 Progress: 1500/3181 bills processed...
📍 Progress: 2000/3181 bills processed...
📍 Progress: 2500/3181 bills processed...
📍 Progress: 3000/3181 bills processed...
📍 Progress: 3181/3181 bills processed...

Roll Call Scan Summary:
hasRollCall
None    3121
True      60
Name: count, dtype: int64

Example preview:
  billNumber rollCallNumbers hasRollCall
0      S1160     

In [ ]:
#df_legislators_192["Name"].unique()

In [ ]:
df_vote_details_191, df_legislators_191 = build_session_data(session_number=191)


Building data for General Court 191 ...

 Fetching documents from General Court 191 ...
Successfully retrieved 9690 documents from General Court 191.
Extracted 9690 entries from documents for General Court 191...

Missing value summary:
billNumber          1437
docketNumber        1157
title                  0
branch              1437
isDocketBookOnly       0
dtype: int64

 Count by branch:
branch
House     5248
Senate    3005
None      1437
Name: count, dtype: int64

Starting roll call scan for General Court 191
Total bills to scan: 3005

📍 Progress: 500/3005 bills processed...
📍 Progress: 1000/3005 bills processed...
📍 Progress: 1500/3005 bills processed...
📍 Progress: 2000/3005 bills processed...
📍 Progress: 2500/3005 bills processed...
📍 Progress: 3000/3005 bills processed...
📍 Progress: 3005/3005 bills processed...

Roll Call Scan Summary:
hasRollCall
None    2938
True      67
Name: count, dtype: int64

Example preview:
  billNumber rollCallNumbers hasRollCall
0      S2065       

##### Merge datasets


In [ ]:
def merge_legislator_votes(vote_file, legislator_file, output_file=None):
    """
    Merge the legislator info sheet with the voting details sheet into a single comprehensive dataset.
    """
    if isinstance(vote_file, str):
        votes = pd.read_csv(vote_file)
    else:
        votes = vote_file.copy()

    if isinstance(legislator_file, str):
        members = pd.read_csv(legislator_file)
    else:
        members = legislator_file.copy()

    print(f"Loaded {len(votes)} vote records and {len(members)} legislators.")

    # Remove duplicate branch to avoid conflicts
    votes = votes.drop(columns=["Branch"], errors="ignore")

    merged = pd.merge(
        members,
        votes,
        on="MemberCode",
        how="left"  
    )

    columns = [
        "GeneralCourtNumber", "MemberCode", "Name", "Branch", 
        "BillNumber", "RollCallNumber", "QuestionMotion", "Vote", "Party", "District",
        "LeadershipPosition", "EmailAddress", "PhoneNumber", "RoomNumber", "MemberUrl"
    ]
    merged = merged[[c for c in columns if c in merged.columns]]

    # Sorted by Member and Voting Order
    merged["order"] = merged.index
    merged = merged.sort_values(by=["order", "RollCallNumber"], ascending=[True, True])
    merged = merged.drop(columns="order")

    print(f"Merged {len(merged)} rows.")
    return merged



In [ ]:
merged_df = merge_legislator_votes(df_vote_details_193, df_legislators_193)
merged_df.head()

Loaded 6242 vote records and 41 legislators.
Merged 6242 rows.


,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,193,PWM0,Paul W. Mark,Senate,S2251,58,None,Yea,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
1,193,PWM0,Paul W. Mark,Senate,S17,2,Question on adoption of the amendment 1,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
2,193,PWM0,Paul W. Mark,Senate,S17,3,Question on adoption of the amendment 2,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
3,193,PWM0,Paul W. Mark,Senate,S17,4,Question on adoption of the amendment 5,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
4,193,PWM0,Paul W. Mark,Senate,S17,5,Question on adoption of the amendment 6,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...


In [ ]:
merged_df.to_csv("193_merged_legislator_votes.csv", index=False)


In [ ]:
df_vote_details_193.head()

,GeneralCourtNumber,Branch,RollCallNumber,QuestionMotion,BillNumber,MemberCode,Vote,MemberUrl
0,193,Senate,58,None,S2251,PWM0,Yea,http://malegislature.gov/api/GeneralCourts/193...
1,193,Senate,58,None,S2251,PMP0,Yea,http://malegislature.gov/api/GeneralCourts/193...
2,193,Senate,58,None,S2251,RKK0,Yea,http://malegislature.gov/api/GeneralCourts/193...
3,193,Senate,58,None,S2251,JRO0,Yea,http://malegislature.gov/api/GeneralCourts/193...
4,193,Senate,58,None,S2251,L M0,Yea,http://malegislature.gov/api/GeneralCourts/193...


In [ ]:
print(df_vote_details_193.columns.tolist())
print(df_legislators_193.columns.tolist())


['GeneralCourtNumber', 'Branch', 'RollCallNumber', 'QuestionMotion', 'BillNumber', 'MemberCode', 'Vote', 'MemberUrl']
['MemberCode', 'Name', 'Branch', 'Party', 'District', 'LeadershipPosition', 'EmailAddress', 'PhoneNumber', 'RoomNumber']


##### Bill Committee Names

Add a bill's committee name to the bills using the general court and pd dataframe.

In [ ]:
#Dataframe I tested it on:
#df = pd.read_csv("194_senate_bill_full_details.csv")

In [ ]:
#Defining functions.

def fetch_all_committees(GENERAL_COURT):
    """Fetch list of all committees"""
    url = f"https://malegislature.gov/api/GeneralCourts/{GENERAL_COURT}/Committees"
    print(f"Fetching committees from: {url}\n")
    
    try:
        response = requests.get(url, headers={'Accept': 'application/json'})
        response.raise_for_status()
        committees = response.json()
        print(f"Found {len(committees)} committees\n")
        return committees
        
    except Exception as e:
        print(f"Error fetching committees: {e}")
        return []


def fetch_committee_details(committee_code,GENERAL_COURT):
    """Fetch detailed information for a specific committee"""
    url = f"https://malegislature.gov/api/GeneralCourts/{GENERAL_COURT}/Committees/{committee_code}"
    
    try:
        response = requests.get(url, headers={'Accept': 'application/json'})
        response.raise_for_status()
        return response.json()
        
    except Exception as e:
        print(f"Error fetching {committee_code}: {e}")
        return None


def get_bill_type(bill_number):
    """Extract bill type from bill number prefix"""
    if not bill_number:
        return "Unknown"
    
    prefix = ''.join([c for c in bill_number if c.isalpha()])
    
    type_map = {
        'H': 'House Bill',
        'S': 'Senate Bill',
        'HR': 'House Resolution',
        'SR': 'Senate Resolution',
        'HD': 'House Document',
        'SD': 'Senate Document'
    }
    
    return type_map.get(prefix, f'{prefix} Document')


def extract_committee_info(committee):
    """Extract relevant fields from committee data including bills"""
    
    # Get chairperson - check both House and Senate
    chairperson = ''
    if committee.get('SenateChairperson') and committee['SenateChairperson'].get('MemberCode'):
        chairperson = committee['SenateChairperson']['MemberCode']
    elif committee.get('HouseChairperson') and committee['HouseChairperson'].get('MemberCode'):
        chairperson = committee['HouseChairperson']['MemberCode']
    
    # Base committee info
    base_info = {
        'Branch': committee.get('Branch', ''),
        'CommitteeCode': committee.get('CommitteeCode', ''),
        'Chairperson': chairperson,
        'ShortName': committee.get('ShortName', '')
    }
    
    # Get bills from committee
    bills = []
    
    # Documents before committee
    docs_before = committee.get('DocumentsBeforeCommittee', [])
    if docs_before:
        for doc in docs_before:
            bill_number = doc.get('DocumentNumber') or doc.get('BillNumber', '')
            if bill_number:
                bills.append({
                    **base_info,
                    'BillNumber': bill_number,
                    'BillType': get_bill_type(bill_number),
                    'Status': 'Before Committee'
                })
    
    # Reported out documents
    reported_out = committee.get('ReportedOutDocuments', [])
    if reported_out:
        for doc in reported_out:
            bill_number = doc.get('DocumentNumber') or doc.get('BillNumber', '')
            if bill_number:
                bills.append({
                    **base_info,
                    'billNumber': bill_number,
                    'BillType': get_bill_type(bill_number),
                    'Status': 'Reported Out'
                })
    
    # If no bills found, return committee info without bills
    if not bills:
        return [{
            **base_info,
            'BillNumber': '',
            'BillType': '',
            'Status': ''
        }]
    
    return bills



In [ ]:
def committeeDataCollection(GENERAL_COURT = 194):
    GENERAL_COURT = str(GENERAL_COURT)

    # Fetch all committees
    committees = fetch_all_committees(GENERAL_COURT)

    if not committees:
        print("No committees found!")
        exit()

    # Extract committee codes
    committee_codes = [c.get('CommitteeCode') for c in committees if c.get('CommitteeCode')]

    # Fetch details for each committee
    all_details = []
        
    for i, code in enumerate(committee_codes, 1):
        #print(f"[{i}/{len(committee_codes)}] Fetching {code}...", end=" ")
        
        details = fetch_committee_details(code, GENERAL_COURT)
        
        if details:
            committee_rows = extract_committee_info(details)
            all_details.extend(committee_rows)  # extend instead of append since we get a list
            #print(f"✓ ({len(committee_rows)} bills)")
        else:
            print(f"✗ at {code}")
        
        # Be respectful to the API
        time.sleep(0.3)

    # Create DataFrame
    df = pd.DataFrame(all_details)
    df.dropna(subset=['billNumber'], axis=0, inplace=True)
    df.rename({"ShortName": "CommitteeName"}, inplace=True)
    df.reset_index(inplace=True, drop=True) #Just resetting index so it doesn't start at random numbers
    # Reorder columns
    df = df[['Branch', 'CommitteeCode', 'Chairperson', 'CommitteeName', 'billNumber', 'BillType', 'Status']]

    #Big text block saying steps. Can be deleted at some point when everything is done.
    print("=" * 80)
    print("MA Legislature Committee Data Collection")
    print("=" * 80)
    print()
    
    print("\n" + "=" * 80)
    print(f"Successfully fetched {len(all_details)} committee-bill records")
    print("=" * 80)
    # Print DataFrame
    print("COMMITTEE DATA:")
    print("=" * 80)
    print(df.head())
    print("\n")

    # Show statistics
    print("SUMMARY:")
    print("=" * 80)
    print(f"Total Records: {len(df)}")
    print(f"Unique Committees: {df['CommitteeCode'].nunique()}")
    print(f"Total Bills: {df[df['billNumber'] != '']['billNumber'].nunique()}")
    print(f"\nBy Branch:")
    print(df['Branch'].value_counts().to_string())
    print(f"\nBy Bill Type:")
    bill_types = df[df['BillType'] != '']['BillType'].value_counts()
    if not bill_types.empty:
        print(bill_types.to_string())
    else:
        print("No bills found")
    # Save to CSV
    #output_file = f"ma_legislature_committees_{GENERAL_COURT}.csv"
    #df.to_csv(output_file, index=False)
    #print(f"\n✓ Saved to: {output_file}")
    return df


In [43]:
cdf = committeeDataCollection(194)

MA Legislature Committee Data Collection

Fetching committees from: https://malegislature.gov/api/GeneralCourts/194/Committees

Found 56 committees


Successfully fetched 9920 committee-bill records

COMMITTEE DATA:
     Branch CommitteeCode Chairperson           ShortName billNumber  \
448  Senate           S30        MJR0  Ways and Means (S)      H4010   
449  Senate           S30        MJR0  Ways and Means (S)      H4001   
450  Senate           S30        MJR0  Ways and Means (S)      S2516   
451  Senate           S30        MJR0  Ways and Means (S)      S2521   
452  Senate           S30        MJR0  Ways and Means (S)      H4206   

        BillType        Status  
448   House Bill  Reported Out  
449   House Bill  Reported Out  
450  Senate Bill  Reported Out  
451  Senate Bill  Reported Out  
452   House Bill  Reported Out  


SUMMARY:
Total Records: 3824
Unique Committees: 36
Total Bills: 3008

By Branch:
Branch
Joint                       2997
House of Representatives     7

In [ ]:
def addCommitteeName(df, cdf):
    """ Adds the committees as a column to the dataframe.
    cdf is the committee data while df is the bill data."""
    df["CommitteeName"] = ""
        
    for i in range(df["billNumber"].size):
        df.loc[i, "CommitteeName"] = cdf["CommitteeName"][(cdf["billNumber"] == df["billNumber"][i]).idxmax()]
    return df
        
    

In [45]:
df2 = addCommitteeName(df,cdf)

In [46]:
print(df2.head())

  billNumber                                              Title  \
0        S14  Proposed Senate Rules for the 2025-2026 Legisl...   
1        S15  Proposed Joint Rules for the Senate and House ...   
2        H58  An Act making appropriations for fiscal year 2...   
3        S16      Fiscal Year 2025 Supplemental Appropriations    
4        S19  Senate Rules for the 2025-2026 Legislative Ses...   

  LegislationTypeName                                        BillHistory  \
0              Report  http://malegislature.gov/api/GeneralCourts/194...   
1              Report  http://malegislature.gov/api/GeneralCourts/194...   
2                Bill  http://malegislature.gov/api/GeneralCourts/194...   
3           Amendment  http://malegislature.gov/api/GeneralCourts/194...   
4              Report  http://malegislature.gov/api/GeneralCourts/194...   

                                             Pinslip  \
0  Senate, February 6, 2025 – Text of the propose...   
1  Senate, February 6, 2026 

In [47]:
df2.to_csv("194_senate_bill_with_committee.csv")